# Evaluating AI Agents
## A Hands-On Workshop

Welcome! In this workshop, you'll learn how to design evaluations for AI agents.

## Two "Axes" of Evaluation

When evaluating AI agents, we think along **two axes**:

*Inspired by [Andrew Ng's Agentic AI course](https://www.deeplearning.ai/short-courses/)*

### Axis 1: Per example ground truth?

- **Per example ground truth**: You know the correct answer for each test case
  - *Example: "Extract the date from this invoice" → expected: "2024-03-15"*
- **No per example ground truth**: There's no single "correct" answer
  - *Example: "Write marketing copy for this product"*

### Axis 2: Evaluate with code or LLM-as-judge?

- **Evaluate with code (objective)**: Code can verify correctness
  - *Example: `if extracted_date == actual_date: num_correct += 1`*
- **LLM-as-judge (subjective)**: Requires an LLM to evaluate quality
  - *Example: "Grade this chart according to whether it has clear axes labels..."*

### The Evaluation Quadrant

These two axes give us four quadrants — each with different evaluation strategies:

![Two Axes of Evaluation](assets/two-axes-of-evaluation.png)

### Today's Focus

We'll explore **two quadrants** with hands-on examples:

**Top-left: SQL Agent**
- Has ground truth (we know the correct query results)
- Evaluate with code (compare actual vs expected)

**Bottom-right: Web Search Agent**
- No ground truth (no single "correct" research summary)
- LLM-as-judge (use a rubric to assess quality)

---
## Setup

Run this cell to import everything we need.

In [ ]:
# Environment
from dotenv import load_dotenv
load_dotenv()

from datetime import datetime
from aisuite import Client

# Utilities
from utils.helpers import make_verbose_tool
from utils.display import print_html, display_rubric_scores, display_test_results
from utils.evaluators import evaluate_sql_result, llm_judge

# Import the tool FUNCTION (aisuite calls this automatically)
from tools.research_tools import tavily_search_tool

# Initialize the LLM client
client = Client()

print("Setup complete!")

### API Key Configuration

This workshop uses two external services:

| Service | Purpose | How to Get |
|---------|---------|------------|
| **Tavily** | Web search tool | Free at [tavily.com](https://tavily.com) |
| **Azure OpenAI** | LLM for agents | Ask Simon |
| **OpenAI** *(optional)* | LLM alternative | [platform.openai.com](https://platform.openai.com/api-keys) |

Run the cell below to check your keys are configured:

In [ ]:
# Check API keys and detect which LLM provider to use
from utils.helpers import check_api_keys
MODEL = check_api_keys()

---
## SQL Agent — Objective Evaluation

Let's start with the simplest type of evaluation: **objective with ground truth**.

We have a SQL agent that answers questions about a product nutrition database.

The evaluation is simple: **does the query return the expected result?**

### The Database

Our database contains nutrition information for food products:
- `product_name` - Product name
- `kcal_per_100g`, `protein_per_100g`, `sugar_per_100g`, etc.
- `npm_score` - Nutrient Profile Model score (UK HFSS metric)

In [ ]:
# Set up the products database
from tools.sql_tools import db, setup_products_db

count = setup_products_db()
print(f"Database ready! {count} products loaded.")

# Quick peek at the data
db.execute("SELECT product_name, kcal_per_100g, protein_per_100g, sugar_per_100g FROM products LIMIT 5").fetchdf()

### The Tool

A **tool** is just a function that the LLM can call. Here's our SQL query tool:

In [ ]:
# ===========================================
# THE SQL QUERY TOOL
# ===========================================
# This is what the agent will use to execute SQL queries.
# A tool is just a function that takes arguments and returns results!

def sql_query_tool(query: str) -> list[dict]:
    """Execute a SQL query and return results as list of dicts."""
    try:
        result = db.execute(query).fetchdf()
        return result.to_dict(orient="records")
    except Exception as e:
        return [{"error": str(e)}]

print("Tool defined: sql_query_tool(query) -> list[dict]")

### Computing Ground Truth

Before we can test the agent, we need to know the **correct answers**.

Let's run SQL queries directly on the database to establish our ground truth values:

In [ ]:
# Q1: How many products are above the target converted NPM score threshold of 69?
sql_query_tool("""
    SELECT COUNT(*) as count 
    FROM products 
    WHERE converted_npm_score > 69
""")

In [ ]:
# Q2: What percentage of products are above the threshold?
sql_query_tool("""
    SELECT ROUND(100.0 * COUNT(*) FILTER (WHERE converted_npm_score > 69) / COUNT(*), 1) as pct
    FROM products
""")

In [ ]:
# Q3: Which product has the highest absolute protein content?
sql_query_tool("""
    SELECT product_name, ROUND(protein_per_100g * grams / 100, 1) as total_protein_g
    FROM products 
    ORDER BY total_protein_g DESC 
    LIMIT 1
""")

In [ ]:
# Q4: What is the average kcal_per_100g across all products?
sql_query_tool("""
    SELECT ROUND(AVG(kcal_per_100g)) as avg_kcal
    FROM products
""")

In [ ]:
# Q5: HARDER - Which products are in the bottom 25% for healthiness but top 25% for protein?
sql_query_tool("""
    WITH ranked AS (
        SELECT product_name, converted_npm_score, protein_per_100g,
            NTILE(4) OVER (ORDER BY converted_npm_score ASC) as health_quartile,
            NTILE(4) OVER (ORDER BY protein_per_100g DESC) as protein_quartile
        FROM products
    )
    SELECT product_name, converted_npm_score, protein_per_100g
    FROM ranked
    WHERE health_quartile = 1 AND protein_quartile = 1
""")

In [ ]:
# ===========================================
# THE SQL AGENT
# ===========================================

def sql_agent(question: str, model: str = MODEL) -> dict:
    """
    Answer questions about the products database using SQL.

    Returns:
        dict with 'answer' (extracted value) and 'raw_response' (full LLM output)
    """
    print(f"Question: {question}")

    prompt = f"""
    You have access to a SQL database with a 'products' table containing food nutrition data.

    Columns: product_name, grams, standardised_volume, kcal_per_100g, fat_per_100g,
    satfat_per_100g, protein_per_100g, carbs_per_100g, sugar_per_100g,
    sodium_per_100g, salt_per_100g, fibre_per_100g, npm_score, converted_npm_score

    Answer this question: {question}

    Instructions:
    1. Write a SQL query to get the answer
    2. Execute it using sql_query_tool
    3. Return ONLY the final answer (a number, name, or short phrase)
    """.strip()

    tools = [make_verbose_tool(sql_query_tool, "database")]

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        tools=tools,
        tool_choice="auto",
        max_turns=3,
    )

    answer = response.choices[0].message.content
    print(f"Answer: {answer}")
    return {"answer": answer, "raw_response": response}

print("SQL agent defined!")

### Try it yourself!

Test the agent with your own questions about the products database:

In [ ]:
# ===========================================
# TRY YOUR OWN QUESTION!
# ===========================================
MY_QUESTION = "Which product has the most sugar per 100g?"

result = sql_agent(MY_QUESTION)

### Testing the Agent

Now that we have ground truth, let's see if the agent can figure out the correct SQL queries on its own.

We'll give it the same questions in natural language and check if its answers match:

In [ ]:
# ===========================================
# EVALUATION TEST CASES
# ===========================================
# Each test case has a question and the expected answer

TEST_CASES = [
    {
        "question": "What pct of products are above the target converted npm score threshold of 69?",
        "expected": "36",
        "comparison": "contains"
    },
    {
        "question": "Which product has the highest absolute protein content, and how much is it?",
        "expected": "Porridge Oats",
        "comparison": "contains"
    },
    {
        "question": "What is the average kcal_per_100g across all products? Round to the nearest whole number.",
        "expected": "281",
        "comparison": "contains"
    },
    {
        "question": "Which products are in the bottom 25% for healthiness but top 25% for protein content?",
        "expected": "Cheddar Cheese",
        "comparison": "contains"
    },
]

print(f"Defined {len(TEST_CASES)} test cases")

In [ ]:
# Run all test cases
print("Running all test cases...\n")

results = []
for test in TEST_CASES:
    result = sql_agent(test["question"])
    passed, _ = evaluate_sql_result(
        result["answer"],
        test["expected"],
        test["comparison"]
    )
    results.append({
        "question": test["question"],
        "expected": test["expected"],
        "actual": result["answer"],
        "passed": passed
    })
    print()

# Display results
display_test_results(results, title="SQL Agent Evaluation")

---
### Key Takeaway: SQL Agent Evaluation

The SQL agent evaluation is **objective with ground truth**:
- We know the correct answer
- We can check it with code
- No ambiguity about pass/fail

**But what about subjective qualities?**
- Is the SQL query efficient?
- Is the answer well-formatted?

For these, we'd need **LLM-as-judge** evaluation.

Let's test that out by looking at a more complex agent where evaluation isn't so clear-cut...

---
## The Research Agent

We have a research agent that searches the web for information on any topic.

### How it works

The agent uses an LLM to call a tool:
- **tavily_tool** - general web search  

**You could design your own tools and plug them in!**

In [ ]:
# ===========================================
# THE RESEARCH AGENT
# ===========================================
# This function orchestrates an LLM with a web search tool to research any topic.
# You could modify this to use different tools or prompts!

def research_agent(topic: str, model: str = MODEL) -> str:
    """
    Research a topic using web search.
    
    Args:
        topic: The topic to research
        model: The LLM to use for orchestration
    
    Returns:
        Research results with sources and URLs
    """
    print(f"Researching: {topic}")
    
    prompt = f"""
    You are a research assistant. Use tavily_search_tool ONCE to search for information, 
    then immediately provide a summary based on what you found.

    Topic to research: {topic}

    Instructions:
    1. Search once for the topic
    2. Summarize the key findings (2-3 paragraphs)
    3. List the source URLs at the end

    Today's date is {datetime.now().strftime('%Y-%m-%d')}.
    """.strip()

    tools = [make_verbose_tool(tavily_search_tool, "web")]

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        tools=tools,
        tool_choice="auto",
        max_turns=2,  # 1 search + 1 response
    )
    
    print("Done!")
    return response.choices[0].message.content

print("Research agent defined!")

### Try it yourself

Change the `TOPIC` variable below to research anything you're curious about!

In [ ]:
# ===========================================
# CHANGE THIS to research your own topic!
# ===========================================
TOPIC = "recent developments in large language models"

print(f"Topic: {TOPIC}")

In [ ]:
# Run the research agent (this may take 10-30 seconds)
research_output = research_agent(TOPIC)

print_html(research_output, title=f"Research Results: {TOPIC}")

---
## LLM-as-Judge Evaluation

Some qualities are **subjective** and hard to check with code:
- Is the writing clear and well-organized?
- Are the sources relevant to the query?
- Is the information comprehensive?

**Solution**: Use another LLM as a "judge" to score the output against a rubric.

### The Judge Prompt

This is the actual prompt sent to the judge LLM. You can modify it!

### How it works

1. Define a **rubric** with scoring dimensions
2. Build a **prompt** that asks the LLM to score each dimension
3. Parse the structured JSON response

In [ ]:
# ===========================================
# THE JUDGE PROMPT
# ===========================================
# This is the prompt sent to the judge LLM. Try modifying it!
# - Change the calibration line to see how scores change
# - Add more specific instructions
# - Change the scoring scale

JUDGE_PROMPT_TEMPLATE = """You are an evaluation judge. Score the following output on each dimension.

## Output to Evaluate:
{output}

## Scoring Rubric (score each 1-5):
{rubric_text}

## Instructions:
For each dimension, provide:
1. A score from 1 (poor) to 5 (excellent)
2. A brief explanation (1-2 sentences)

Be critical. A score of 5 should be rare.

Respond in JSON format:
{{
    "scores": {{{score_template}}},
    "explanations": {{{explanation_template}}}
}}
"""

print("Judge prompt template defined!")

### Activity: Design a Rubric

A rubric defines what dimensions to evaluate. **This is a design choice** — there's no single "right" rubric!

We'll start with one criterion, then you'll add two more based on what *you* think matters.

In [ ]:
# ===========================================
# ACTIVITY: Design your rubric!
# ===========================================
# We've started with ONE criterion. Add 2 more based on what YOU think matters.
# Ask yourself: "If I were grading this summary, what would I look for?"

RUBRIC = {
    "coherence": "Does the summary flow logically? Are ideas connected with smooth transitions?",
    
    # ADD 2 MORE CRITERIA BELOW!
    # Examples to consider:
    #   "clarity": "Is the writing clear and easy to understand?"
    #   "completeness": "Does it cover the key aspects of the topic?"
    #   "relevance": "Does it stay focused on the research question?"
    #   "accuracy": "Are claims supported by the sources cited?"
    #   "conciseness": "Is it appropriately brief without unnecessary filler?"
}

print(f"Rubric has {len(RUBRIC)} dimension(s)")
for dim, desc in RUBRIC.items():
    print(f"  - {dim}: {desc}")

In [ ]:

# Build the prompt from template + rubric + output
rubric_text = "\n".join(f"- **{dim}**: {desc}" for dim, desc in RUBRIC.items())
score_template = ", ".join(f'"{dim}": <score>' for dim in RUBRIC.keys())
explanation_template = ", ".join(f'"{dim}": "<explanation>"' for dim in RUBRIC.keys())

judge_prompt = JUDGE_PROMPT_TEMPLATE.format(
    output=research_output,
    rubric_text=rubric_text,
    score_template=score_template,
    explanation_template=explanation_template
)

# Run the LLM judge evaluation
judge_result = llm_judge(judge_prompt, RUBRIC)

# Display the results
display_rubric_scores(judge_result, title="LLM Judge Evaluation")

---
## Human Review Patterns

Sometimes you need **human judgment** that can't be automated:
- Is this factually accurate? (requires domain expertise)
- Is this appropriate for our use case?
- Would a user find this helpful?

### When to use each method?

| Method | Use When |
|--------|----------|
| **Automated** | Objective, code-checkable criteria |
| **LLM-as-judge** | Subjective but definable in a rubric |
| **Human review** | Domain expertise required, high stakes |

### Example: Human Review Checklist

```python
HUMAN_REVIEW_CHECKLIST = {
    "factual_accuracy": {
        "question": "Are all factual claims accurate?",
        "type": "yes_no_unsure",
    },
    "source_verification": {
        "question": "Did you spot-check at least one source URL?",
        "type": "yes_no",
    },
    "user_helpful": {
        "question": "Would this be helpful to someone researching this topic?",
        "type": "scale_1_5",
    },
}
```

### Key principles for human review

1. **Structure the review** - Don't ask "is this good?", ask specific questions
2. **Use checklists** - Binary yes/no questions are faster and more consistent
3. **Provide context** - Show reviewers what they need to evaluate
4. **Collect rationale** - Ask reviewers to explain their judgments

---
## Putting It Together

In practice, you'll often combine multiple evaluation types:

```
Agent Output
    │
    ▼
┌─────────────────┐
│ Objective Checks│ ──► Fast, cheap, catches obvious issues
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│   LLM Judge     │ ──► Scalable quality assessment  
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  Human Review   │ ──► High-stakes verification (sample)
└─────────────────┘
```

### The Key Insight

**Layer your evaluations by cost and coverage:**

| Method | Cost | Coverage | Best For |
|--------|------|----------|----------|
| Objective checks | Low | 100% | Catching obvious failures |
| LLM-as-judge | Medium | 100% | Quality assessment at scale |
| Human review | High | Sample | Validation, edge cases |

---
## Key Takeaways

1. **Component-level evals** let you test individual pieces of an AI system

2. **Objective evaluations** (ground truth) are fast, cheap, reproducible

3. **LLM-as-judge** scales subjective evaluation but requires rubric design

4. **Human review** is essential for high-stakes and domain expertise

5. **Combine methods** for robust evaluation pipelines

---
## Next Steps

To build a proper evaluation suite:

1. **Create an eval set**: 10-20 diverse test cases
2. **Define ground truth**: What should pass/fail?
3. **Run evals regularly**: Track metrics over time
4. **Iterate**: Improve your agent based on results